# Merging & Target Variable Construction

Constructs trading-day event windows around each Resolution for the 10-year G-Sec yield and USD/INR, and extracts a repo-rate-change control variable directly from each Resolution's text.

## Step 0 — Path setup

In [1]:
import sys
import re
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\hp\Desktop\rbi-sentiment-market-forecast


## Step 1 — Inspect the G-Sec source file

Confirms column names and date format prior to parsing.

In [2]:
from paths import GSEC_CSV

gsec_raw = pd.read_csv(GSEC_CSV)
print("Columns:", gsec_raw.columns.tolist())
print()
print(gsec_raw.head())
print()
print(gsec_raw.dtypes)

Columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Change %']

         Date  Price   Open   High    Low Change %
0  19-08-2026  6.830  6.826  6.832  6.816   -0.01%
1  18-08-2026  6.831  6.830  6.851  6.823    0.40%
2  17-08-2026  6.804  6.801  6.819  6.798    0.62%
3  14-08-2026  6.762  6.757  6.766  6.749    0.06%
4  13-08-2026  6.758  6.777  6.777  6.753   -0.28%

Date         object
Price       float64
Open        float64
High        float64
Low         float64
Change %     object
dtype: object


## Step 2 — Parse the G-Sec yield series

In [3]:
date_col = "Date"
price_col = "Price"

gsec = gsec_raw[[date_col, price_col]].copy()
gsec[date_col] = pd.to_datetime(gsec[date_col], dayfirst=True, errors="coerce")
gsec[price_col] = gsec[price_col].astype(str).str.replace(",", "", regex=False).astype(float)
gsec = gsec.dropna(subset=[date_col]).sort_values(date_col).rename(columns={date_col: "date", price_col: "gsec_yield"})
gsec_series = gsec.set_index("date")["gsec_yield"]

print(f"Parsed {len(gsec_series)} rows, {gsec_series.index.min().date()} to {gsec_series.index.max().date()}")
gsec_series.tail()

Parsed 2439 rows, 2016-09-01 to 2026-08-19


date
2026-08-13    6.758
2026-08-14    6.762
2026-08-17    6.804
2026-08-18    6.831
2026-08-19    6.830
Name: gsec_yield, dtype: float64

## Step 3 — Load the USD/INR series

In [4]:
from paths import USDINR_CSV

fx = pd.read_csv(USDINR_CSV, parse_dates=["date"]).sort_values("date")
fx_series = fx.set_index("date")["usdinr"]
print(f"Loaded {len(fx_series)} rows, {fx_series.index.min().date()} to {fx_series.index.max().date()}")

Loaded 2592 rows, 2016-09-01 to 2026-08-19


## Step 4 — Load sentiment scores (Resolutions only)

In [5]:
from paths import SENTIMENT_CSV

sentiment = pd.read_csv(SENTIMENT_CSV, parse_dates=["listed_date"])
resolutions = sentiment[sentiment["doc_type"] == "resolution"].sort_values("listed_date").reset_index(drop=True)
print(f"{len(resolutions)} Resolutions")
resolutions[["listed_date", "lexicon_score_x1000", "finbert_score"]].head()

61 Resolutions


,listed_date,lexicon_score_x1000,finbert_score
0,2016-10-04,-3.331113,0.055696
1,2016-12-07,0.413907,-0.027087
2,2017-02-08,-1.378043,0.111587
3,2017-04-06,-2.047083,0.126439
4,2017-06-07,-1.280820,0.053529


## Step 5 — Event-window construction

For each Resolution date, locates the nearest trading day at or before the meeting, then computes yield/FX values at trading-day offsets of −1, +1, and +3 — robust to weekends and market holidays.

In [6]:
def event_window_values(event_date, series, offsets=(-1, 1, 3)):
    idx = series.index
    pos = idx.searchsorted(event_date, side="right") - 1
    out = {}
    for o in offsets:
        p = pos + o
        out[o] = series.iloc[p] if 0 <= p < len(series) else None
    return out

rows = []
for _, row in resolutions.iterrows():
    d = row["listed_date"]
    y = event_window_values(d, gsec_series)
    f = event_window_values(d, fx_series)
    rows.append({
        "prid": row["prid"],
        "listed_date": d,
        "lexicon_score_x1000": row["lexicon_score_x1000"],
        "finbert_score": row["finbert_score"],
        "gsec_yield_tm1": y[-1], "gsec_yield_tp1": y[1], "gsec_yield_tp3": y[3],
        "dyield_1d": (y[1] - y[-1]) if None not in (y[1], y[-1]) else None,
        "dyield_3d": (y[3] - y[-1]) if None not in (y[3], y[-1]) else None,
        "usdinr_tm1": f[-1], "usdinr_tp1": f[1], "usdinr_tp3": f[3],
        "dusdinr_1d_pct": ((f[1] - f[-1]) / f[-1] * 100) if None not in (f[1], f[-1]) else None,
        "dusdinr_3d_pct": ((f[3] - f[-1]) / f[-1] * 100) if None not in (f[3], f[-1]) else None,
    })

merged = pd.DataFrame(rows)
print(f"Built event windows for {len(merged)} Resolutions")
print(f"Missing dyield_1d: {merged['dyield_1d'].isna().sum()}")
merged[["listed_date", "lexicon_score_x1000", "dyield_1d", "dyield_3d", "dusdinr_1d_pct"]].head(10)

Built event windows for 61 Resolutions
Missing dyield_1d: 0


,listed_date,lexicon_score_x1000,dyield_1d,dyield_3d,dusdinr_1d_pct
0,2016-10-04,-3.331113,-0.102,-0.051,0.047338
1,2016-12-07,0.413907,0.199,0.222,-0.854245
2,2017-02-08,-1.378043,0.424,0.397,-0.343767
3,2017-04-06,-2.047083,0.168,0.160,-0.722523
4,2017-06-07,-1.280820,-0.103,-0.120,0.115465
5,2017-08-02,-0.984252,-0.011,0.017,-0.855125
6,2017-10-04,-0.335683,0.081,0.132,-0.842132
7,2017-12-06,-2.409639,-0.005,0.113,0.254533
8,2018-02-07,-2.478315,-0.102,-0.072,-0.178342
9,2018-04-05,-0.700771,-0.119,0.085,-0.067014


## Step 6 — Repo-rate-change extraction

Extracted via pattern matching on the Resolution text (`extract_repo_rate()`, `src/sentiment.py`-adjacent logic). Validated against six independently known historical events: the 2018 hiking cycle, 2019 cutting cycle, the March 2020 emergency COVID cut, the 2020-22 pandemic-era hold, and the 2022 hiking cycle.

In [7]:
from paths import MASTER_TEXT_CSV

In [8]:
_REPO_LEVEL_RE = re.compile(r"policy repo rate.{0,80}?(?:to|at)\s+(\d+\.\d+)\s+per\s+cent", re.IGNORECASE | re.DOTALL)
_REPO_BPS_RE = re.compile(r"by\s+(\d+)\s+basis\s+points?", re.IGNORECASE)
_REPO_DIRECTION_RE = re.compile(r"\b(increase|decrease|reduce|raise|cut)\b", re.IGNORECASE)

def extract_repo_rate(text):
    m = _REPO_LEVEL_RE.search(text)
    if not m:
        return None, None
    level = float(m.group(1))
    window = text[max(0, m.start()-60):m.end()]
    bps_m = _REPO_BPS_RE.search(window)
    dir_m = _REPO_DIRECTION_RE.search(window)
    bps = int(bps_m.group(1)) if bps_m else 0
    if dir_m and dir_m.group(1).lower() in ("decrease", "reduce", "cut"):
        bps = -bps
    elif not dir_m:
        bps = 0  # no direction word found -> treat as unchanged
    return level, bps

master = pd.read_csv(MASTER_TEXT_CSV)
res_text = master[master["doc_type"] == "resolution"][["prid", "text"]]

repo_rows = []
for _, row in res_text.iterrows():
    level, bps = extract_repo_rate(row["text"])
    repo_rows.append({"prid": row["prid"], "repo_rate_level": level, "repo_rate_change_bps": bps})

repo_df = pd.DataFrame(repo_rows)
merged = merged.merge(repo_df, on="prid", how="left")
print(merged[["listed_date", "repo_rate_level", "repo_rate_change_bps"]].to_string())

   listed_date  repo_rate_level  repo_rate_change_bps
0   2016-10-04              NaN                   NaN
1   2016-12-07             6.25                   0.0
2   2017-02-08             6.25                   0.0
3   2017-04-06             6.25                   0.0
4   2017-06-07             6.25                   0.0
5   2017-08-02              NaN                   NaN
6   2017-10-04             6.00                   0.0
7   2017-12-06             6.00                   0.0
8   2018-02-07             6.00                   0.0
9   2018-04-05             6.00                   0.0
10  2018-06-06             6.25                  25.0
11  2018-08-01             6.50                  25.0
12  2018-10-05             6.50                   0.0
13  2018-12-05             6.50                   0.0
14  2019-02-07              NaN                   NaN
15  2019-04-04             6.00                 -25.0
16  2019-06-06             5.75                 -25.0
17  2019-08-07              

## Step 7 — Persist the merged dataset

In [9]:
from paths import MERGED_CSV

merged.to_csv(MERGED_CSV, index=False)
print(f"Saved {len(merged)} rows -> {MERGED_CSV}")
merged.describe()

Saved 61 rows -> C:\Users\hp\Desktop\rbi-sentiment-market-forecast\data\processed\merged_dataset.csv


,prid,listed_date,lexicon_score_x1000,finbert_score,gsec_yield_tm1,gsec_yield_tp1,gsec_yield_tp3,dyield_1d,dyield_3d,usdinr_tm1,usdinr_tp1,usdinr_tp3,dusdinr_1d_pct,dusdinr_3d_pct,repo_rate_level,repo_rate_change_bps
count,61.000000,61,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,61.000000,57.000000,57.000000
mean,51871.639344,2021-09-09 04:43:16.721311488,-1.547300,0.114470,6.804557,6.829787,6.847393,0.025230,0.042836,76.972880,76.902380,77.039800,-0.093632,0.083705,5.512281,0.877193
min,38224.000000,2016-10-04 00:00:00,-8.567931,-0.266591,5.826000,5.886000,5.900000,-0.184000,-0.127000,64.189499,63.640598,63.665100,-1.262623,-1.389385,4.000000,-75.000000
25%,46722.000000,2019-04-04 00:00:00,-4.088785,0.030986,6.431000,6.429000,6.457000,-0.048000,-0.041000,71.099998,70.870003,71.035004,-0.345249,-0.240197,4.900000,0.000000
50%,52366.000000,2021-10-08 00:00:00,-1.236476,0.126439,6.806000,6.797000,6.811000,0.018000,0.029000,75.391296,75.464996,75.685402,-0.091756,0.045189,5.900000,0.000000
75%,57275.000000,2024-02-08 00:00:00,0.000000,0.214937,7.178000,7.227000,7.259000,0.066000,0.103000,83.373299,83.296799,83.250298,0.168692,0.339026,6.500000,0.000000
max,63287.000000,2026-08-05 00:00:00,6.639004,0.393368,8.158000,7.993000,8.031000,0.424000,0.397000,96.164497,95.077301,95.360298,1.531116,1.978721,6.500000,50.000000
std,6940.074685,NaN,3.349651,0.137706,0.524788,0.523767,0.532303,0.110914,0.110874,8.458650,8.458094,8.492296,0.478475,0.622690,0.955449,20.444236


---

Output: `data/processed/merged_dataset.csv`, one row per Resolution (n=61), with sentiment scores, yield/FX event-window changes, and the repo-rate control. Used directly by the econometric analysis (`06_econometrics.ipynb`).